# ABC Vincent: selected simulated scenarios

This notebook is a clean driver for the simulated experiment.

It does not contain the ABC algorithm itself. It calls the reusable Python file:

`simulations_abc/terminal_running/run_simulated_9cases_parallel.py`

You can run:

- one scenario, for example `z_90` with `size_80`
- a subset, for example `z_90`, `z_80` with `size_80`
- all 9 scenarios

The output is saved in `simulations_abc/jupyters/artifacts/`.


In [9]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        runner = path / "simulations_abc" / "terminal_running" / "run_simulated_9cases_parallel.py"
        if runner.exists():
            return path
    raise RuntimeError("Could not find project root.")


PROJECT_ROOT = find_project_root()
RUNNER_PATH = PROJECT_ROOT / "simulations_abc" / "terminal_running" / "run_simulated_9cases_parallel.py"
ARTIFACTS_DIR = PROJECT_ROOT / "simulations_abc" / "jupyters" / "artifacts"
PYTHON_EXE = PROJECT_ROOT / ".venv" / "bin" / "python"
if not PYTHON_EXE.exists():
    PYTHON_EXE = Path(sys.executable)

print("project root:", PROJECT_ROOT)
print("runner:", RUNNER_PATH)
print("python:", PYTHON_EXE)


project root: /home/bardia/projects/graphical-sampling
runner: /home/bardia/projects/graphical-sampling/simulations_abc/terminal_running/run_simulated_9cases_parallel.py
python: /home/bardia/projects/graphical-sampling/.venv/bin/python


## Settings

Change only this cell for most runs.

Examples:

```python
Z_VARIABLES = ["z_90", "z_80"]
SIZE_VARIABLES = ["size_80"]
```

Use `"all"` to run all variables.

Important objective choices:

- `"eff_z"`: focus only on improving `z`
- `"min_relative"`: try to improve `z` and `y` together


In [10]:
# Select scenarios. Use "all" or a list.
Z_VARIABLES = "all"
SIZE_VARIABLES = "all"

# Use "eff_z" if you want to focus on z only.
# Use "min_relative" if you want to protect/improve both z and y.
OBJECTIVE = "eff_z"

ITERATIONS = 250
CHECKPOINT_INTERVAL = 50
COLONY_SIZE = 20
WORKERS = 4


def list_to_arg(value):
    if isinstance(value, str):
        return value
    return ",".join(value)


def safe_tag(value):
    return list_to_arg(value).replace(",", "-")


z_arg = list_to_arg(Z_VARIABLES)
size_arg = list_to_arg(SIZE_VARIABLES)
z_tag = "allz" if z_arg == "all" else safe_tag(Z_VARIABLES)
size_tag = "allsizes" if size_arg == "all" else safe_tag(SIZE_VARIABLES)

FINAL_CSV = ARTIFACTS_DIR / f"abc_random_simulated_{z_tag}_{size_tag}_{OBJECTIVE}_{ITERATIONS}iter.csv"
LIVE_CSV = ARTIFACTS_DIR / f"abc_random_simulated_live_{z_tag}_{size_tag}_{OBJECTIVE}_{ITERATIONS}iter.csv"

print("z variables:", Z_VARIABLES)
print("size variables:", SIZE_VARIABLES)
print("objective:", OBJECTIVE)
print("final CSV:", FINAL_CSV)
print("live CSV:", LIVE_CSV)


z variables: ['z_90']
size variables: ['size_90']
objective: eff_z
final CSV: /home/bardia/projects/graphical-sampling/simulations_abc/jupyters/artifacts/abc_random_simulated_z_90_size_90_eff_z_250iter.csv
live CSV: /home/bardia/projects/graphical-sampling/simulations_abc/jupyters/artifacts/abc_random_simulated_live_z_90_size_90_eff_z_250iter.csv


## Run

This cell runs the `.py` file and prints the results online.

The script saves a live CSV after every finished case, so if you stop the run, you still keep partial results.


In [11]:
cmd = [
    str(PYTHON_EXE),
    str(RUNNER_PATH),
    "--iterations", str(ITERATIONS),
    "--checkpoint-interval", str(CHECKPOINT_INTERVAL),
    "--colony-size", str(COLONY_SIZE),
    "--workers", str(WORKERS),
    "--objective", OBJECTIVE,
    "--z-vars", z_arg,
    "--size-vars", size_arg,
]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

print("Running command:")
print(" ".join(cmd))
print()

process = subprocess.Popen(
    cmd,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Simulation stopped with return code {return_code}")


Running command:
/home/bardia/projects/graphical-sampling/.venv/bin/python /home/bardia/projects/graphical-sampling/simulations_abc/terminal_running/run_simulated_9cases_parallel.py --iterations 250 --checkpoint-interval 50 --colony-size 20 --workers 4 --objective eff_z --z-vars z_90 --size-vars size_90

Parallel simulated 9-case checkpoint run
workers       = 4
checkpoints   = [50, 100, 150, 200, 250]
colony_size   = 20
objective     = eff_z
base_seed     = 202600384
z variables   = ['z_90']
size variables= ['size_90']
live CSV      = /home/bardia/projects/graphical-sampling/simulations_abc/jupyters/artifacts/abc_random_simulated_live_z_90_size_90_eff_z_250iter.csv

=== checkpoint 50 iterations ===
1/1 iter= 50 z_90 / size_90: ABC z=1.271125, ABC y=0.468293, both=False | saved
Saved checkpoint: /home/bardia/projects/graphical-sampling/simulations_abc/jupyters/artifacts/abc_random_simulated_z_90_size_90_eff_z_50iter.csv

=== checkpoint 100 iterations ===
1/1 iter=100 z_90 / size_90: AB

## Final efficiency table

This table shows the final checkpoint results.

`Start_*` is the common Vincent starting design used by both ABC and Random.

`z_over_Ppi_z` compares to the Vincent/Ppi starting design, so `Start_z_over_Ppi_z = 1`.

`*_y_over_Ppi_y_opt` compares `y` to the Vincent benchmark built from the `y/pi` ordering.

The main columns are:

- `corr_y_z`
- `corr_y_pi`
- `corr_y_over_pi_z_over_pi`
- `Start_z_over_Ppi_z`
- `Start_y_over_Ppi_y_opt`
- `ABC_z_over_Ppi_z`
- `ABC_y_over_Ppi_y_opt`
- `Random_z_over_Ppi_z`
- `Random_y_over_Ppi_y_opt`

Values above `1` mean better than Vincent/P$\pi$.


In [12]:
result_path = FINAL_CSV if FINAL_CSV.exists() else LIVE_CSV
if not result_path.exists():
    raise FileNotFoundError("No result CSV found yet. Run the simulation cell first.")

results = pd.read_csv(result_path)

main_cols = [
    "iterations",
    "z_variable",
    "size_variable",
    "objective",
    "corr_y_z",
    "corr_y_pi",
    "corr_y_over_pi_z_over_pi",
    "Start_z_over_Ppi_z",
    "Start_y_over_Ppi_y_opt",
    "ABC_z_over_Ppi_z",
    "ABC_y_over_Ppi_y_opt",
    "Random_z_over_Ppi_z",
    "Random_y_over_Ppi_y_opt",
    "both_ABC",
    "both_Random",
]

results[main_cols].round(4)


,iterations,z_variable,size_variable,objective,corr_y_z,corr_y_pi,ABC_z_over_Ppi_z,ABC_y_over_Ppi_y,Random_z_over_Ppi_z,Random_y_over_Ppi_y,both_ABC,both_Random
0,250,z_90,size_90,eff_z,0.9,0.8655,1.4949,0.3742,1.0,1.0,False,False


## Best cases

These are the cases where ABC improves both `z` and `y` relative to Vincent/P$\pi$.

If you use `OBJECTIVE = "eff_z"`, it is normal that `z` may improve more than `y`.


In [13]:
best_cases = (
    results.loc[results["both_ABC"], main_cols]
    .sort_values(["ABC_y_over_Ppi_y_opt", "ABC_z_over_Ppi_z"], ascending=False)
)

print("ABC both-improved cases:", len(best_cases), "out of", len(results))
print("Random both-improved cases:", int(results["both_Random"].sum()), "out of", len(results))

best_cases.round(4)


ABC both-improved cases: 0 out of 1
Random both-improved cases: 0 out of 1


,iterations,z_variable,size_variable,objective,corr_y_z,corr_y_pi,ABC_z_over_Ppi_z,ABC_y_over_Ppi_y,Random_z_over_Ppi_z,Random_y_over_Ppi_y,both_ABC,both_Random


## Short interpretation

If `OBJECTIVE = "min_relative"`, the optimizer is asked to improve `z` and `y` together. Therefore the two relative efficiencies may look similar.

If `OBJECTIVE = "eff_z"`, the optimizer focuses on `z`; then the efficiency for `y` may be smaller, unchanged, or even worse.

For your question about why results looked too similar, the first thing to check is the objective function.
